<a href="https://colab.research.google.com/github/Nitin-ops320/cricanalytics-ai/blob/main/cric_analytics_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Force install an older, stable version of MediaPipe that includes mp.solutions
!pip install opencv-python mediapipe==0.10.21 ultralytics openpyxl
print("✅ Libraries updated with stable MediaPipe version!")

✅ Libraries updated with stable MediaPipe version!


In [2]:
import cv2

# PASTE YOUR COPIED VIDEO PATH HERE
video_path = '/content/VID_20260602233712.mp4'

# Open the video file
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("❌ Error: Could not open the video file. Please check the path.")
else:
    # Get basic video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps

    print(f"✅ Video loaded successfully!")
    print(f"🎥 Resolution: {width}x{height}")
    print(f"⚡ Frame Rate: {fps:.2f} FPS")
    print(f"🎞️ Total Frames: {frame_count}")
    print(f"⏱️ Video Duration: {duration:.2f} seconds")

    # Read the first frame just to test extraction
    ret, frame = cap.read()
    if ret:
        print(f"📸 Successfully extracted the first frame. Shape: {frame.shape}")

    # Always release the video object when done
    cap.release()

✅ Video loaded successfully!
🎥 Resolution: 480x864
⚡ Frame Rate: 25.00 FPS
🎞️ Total Frames: 53
⏱️ Video Duration: 2.12 seconds
📸 Successfully extracted the first frame. Shape: (864, 480, 3)


In [3]:
import cv2
import mediapipe as mp

# 1. Initialize MediaPipe Pose and Drawing Utilities
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,       # 0=fast/less accurate, 1=balanced, 2=slow/highly accurate
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 2. Paths for input and output videos
input_video_path = '/content/VID_20260602233712.mp4'  # Make sure this matches your uploaded file name!
output_video_path = '/content/cricket_pose_output.mp4'

# Open the input video
cap = cv2.VideoCapture(input_video_path)
if not cap.isOpened():
    print("❌ Error: Could not open input video.")
    exit()

# Get video properties to configure the output file writer
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for MP4 format

# Initialize VideoWriter to save our annotated video
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print("Processing video frames... This might take a minute depending on video length.")

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # End of video reached

    frame_count += 1

    # MediaPipe requires RGB images, but OpenCV reads in BGR format
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process the frame to detect pose landmarks
    results = pose.process(rgb_frame)

    # If landmarks are detected, draw them onto the frame
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2), # Landmark circles (Green)
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2)  # Connection lines (Red)
        )

    # Write the annotated frame to the output video file
    out.write(frame)

# Clean up resources
cap.release()
out.release()
pose.close()

print(f"✅ Finished processing! Processed {frame_count} frames.")
print(f"📁 Your output video has been saved to: {output_video_path}")

Processing video frames... This might take a minute depending on video length.
✅ Finished processing! Processed 53 frames.
📁 Your output video has been saved to: /content/cricket_pose_output.mp4


In [6]:
import cv2
import mediapipe as mp
import numpy as np

# 1. Function to calculate the angle between three joints
def calculate_angle(a, b, c):
    a = np.array(a) # First point (e.g., Shoulder)
    b = np.array(b) # Mid point (e.g., Elbow)
    c = np.array(c) # End point (e.g., Wrist)

    ba = a - b
    bc = c - b

    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    angle = np.degrees(np.arccos(cosine_angle))
    return int(angle)

# 2. Initialize MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

input_video_path = '/content/VID_20260602233712.mp4' # Match your file name!
cap = cv2.VideoCapture(input_video_path)

print("📐 Analyzing cricket biomechanics frame-by-frame...\n")
print(f"{'Frame':<8} | {'Left Elbow Angle':<18} | {'Left Knee Angle':<16}")
print("-" * 50)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # Direct integer indexing to avoid any library naming bugs
        # Left Arm: Shoulder (11) -> Elbow (13) -> Wrist (15)
        left_shoulder = [landmarks[11].x, landmarks[11].y]
        left_elbow    = [landmarks[13].x, landmarks[13].y]
        left_wrist    = [landmarks[15].x, landmarks[15].y]

        # Left Leg: Hip (23) -> Knee (25) -> Ankle (27)
        left_hip      = [landmarks[23].x, landmarks[23].y]
        left_knee     = [landmarks[25].x, landmarks[25].y]
        left_ankle    = [landmarks[27].x, landmarks[27].y]

        # Calculate angles
        elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
        knee_angle = calculate_angle(left_hip, left_knee, left_ankle)

        # Print metrics for every 5th frame to keep it readable
        if frame_id % 5 == 0:
            print(f"{frame_id:<8} | {elbow_angle:<16}° | {knee_angle:<14}°")

cap.release()
pose.close()
print("\n✅ Biomechanical analysis complete!")

📐 Analyzing cricket biomechanics frame-by-frame...

Frame    | Left Elbow Angle   | Left Knee Angle 
--------------------------------------------------
5        | 147             ° | 114           °
10       | 162             ° | 117           °
15       | 73              ° | 110           °
25       | 163             ° | 178           °
35       | 15              ° | 166           °
40       | 27              ° | 164           °
45       | 129             ° | 160           °
50       | 37              ° | 163           °

✅ Biomechanical analysis complete!


In [7]:
import cv2
import mediapipe as mp
import numpy as np

# 1. Reuse our trusty angle calculation function
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    return int(np.degrees(np.arccos(cosine_angle)))

# 2. Initialize MediaPipe Pose and Drawing setup
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

input_video_path = '/content/VID_20260602233712.mp4'  # Match your filename!
output_video_path = '/content/cricket_telemetry_output.mp4'

cap = cv2.VideoCapture(input_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print("🎬 Rendering real-time biomechanical telemetry overlay...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb_frame)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # Extract coordinates
        left_shoulder = [landmarks[11].x, landmarks[11].y]
        left_elbow    = [landmarks[13].x, landmarks[13].y]
        left_wrist    = [landmarks[15].x, landmarks[15].y]

        left_hip      = [landmarks[23].x, landmarks[23].y]
        left_knee     = [landmarks[25].x, landmarks[25].y]
        left_ankle    = [landmarks[27].x, landmarks[27].y]

        # Calculate angles
        elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
        knee_angle = calculate_angle(left_hip, left_knee, left_ankle)

        # Draw the standard skeleton wireframe first
        mp_drawing.draw_landmarks(
            frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2)
        )

        # Convert normalized coordinates (0 to 1) back to actual video pixels for text placement
        elbow_pixel = (int(left_elbow[0] * width), int(left_elbow[1] * height))
        knee_pixel  = (int(left_knee[0] * width), int(left_knee[1] * height))

        # Render the text boxes onto the frame
        # cv2.putText parameters: (image, text, position, font, scale, color, thickness, line_type)
        cv2.putText(frame, f"{elbow_angle} Deg", (elbow_pixel[0] + 15, elbow_pixel[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.putText(frame, f"{knee_angle} Deg", (knee_pixel[0] + 15, knee_pixel[1]),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA)

    out.write(frame)

cap.release()
out.release()
pose.close()

print(f"✅ Success! Video saved to: {output_video_path}")
print("👉 Go to your files panel, hit refresh, and download 'cricket_telemetry_output.mp4'!")

🎬 Rendering real-time biomechanical telemetry overlay...
✅ Success! Video saved to: /content/cricket_telemetry_output.mp4
👉 Go to your files panel, hit refresh, and download 'cricket_telemetry_output.mp4'!
